In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

In [6]:
df = pd.read_csv("../../data/creditcard.csv")

In [7]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# DIVISÃO TREINO/TESTE
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [8]:
# =========================
# INSTALAR PYCARET
# =========================
#%pip install pycaret

# =========================
# IMPORTS
# =========================
import pandas as pd

from sklearn.model_selection import train_test_split
from pycaret.classification import setup, compare_models, predict_model, pull
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

# =========================
# PYCARET
# =========================

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

exp = setup(
    data=train_df,
    test_data=test_df,
    target="Class",
    index=False,  # PyCaret assigns a fresh index
    session_id=42,
    fix_imbalance=True,  # Handle class imbalance
    fix_imbalance_method='smote',  # Use SMOTE for oversampling
    verbose=False
)


# Retorna os 10 melhores modelos treinados
modelos_treinados = compare_models(n_select=10)

# =========================
# LEADERBOARD INTERNO DO PYCARET
# =========================
print("\n===== LEADERBOARD (Validação PyCaret) =====")
print(pull())

# =========================
# MÉTRICAS DE TODOS OS MODELOS (No Test Set)
# =========================
resultados = []

for model in modelos_treinados:
    model_name = model.__class__.__name__

    # Previsão no conjunto de teste
    preds_df = predict_model(model, data=test_df, verbose=False)

    y_true = test_df['Class'].values
    # No PyCaret 3.x, a coluna com o rótulo previsto é 'prediction_label'
    y_pred = preds_df['prediction_label'].astype(int).values

    # Métricas
    acc = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
    f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    average_precision = round(average_precision_score(y_true, y_pred, pos_label=1, average='weighted'), 4)

    resultados.append({
        "Modelo": model_name,
        "Tipo": model_name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "F1-score Weighted": f1_weighted,
        "average_precision": average_precision
    })

# =========================
# RESULTADOS
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="average_precision", ascending=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.9996,0.9755,0.8220,0.9155,0.8641,0.8639,0.8662,4.6060
rf,Random Forest Classifier,0.9995,0.9660,0.8220,0.8919,0.8531,0.8529,0.8548,96.5140
xgboost,Extreme Gradient Boosting,0.9994,0.9763,0.8271,0.8566,0.8385,0.8382,0.8399,1.0180
lightgbm,Light Gradient Boosting Machine,0.9990,0.9711,0.8271,0.6709,0.7380,0.7375,0.7430,1.3380
dummy,Dummy Classifier,0.9983,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1370
dt,Decision Tree Classifier,0.9977,0.8910,0.7838,0.4177,0.5441,0.5430,0.5707,3.8580
gbc,Gradient Boosting Classifier,0.9938,0.9757,0.8651,0.2002,0.3248,0.3229,0.4141,60.8000
nb,Naive Bayes,0.9922,0.9653,0.7510,0.1500,0.2499,0.2477,0.3333,0.2210
lr,Logistic Regression,0.9891,0.9804,0.8880,0.1257,0.2200,0.2176,0.3314,2.9910
ridge,Ridge Classifier,0.9875,0.9731,0.8222,0.1042,0.1849,0.1824,0.2900,0.1690



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
et                 Extra Trees Classifier    0.9996  0.9755  0.8220  0.9155   
rf               Random Forest Classifier    0.9995  0.9660  0.8220  0.8919   
xgboost         Extreme Gradient Boosting    0.9994  0.9763  0.8271  0.8566   
lightgbm  Light Gradient Boosting Machine    0.9990  0.9711  0.8271  0.6709   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
dt               Decision Tree Classifier    0.9977  0.8910  0.7838  0.4177   
gbc          Gradient Boosting Classifier    0.9938  0.9757  0.8651  0.2002   
nb                            Naive Bayes    0.9922  0.9653  0.7510  0.1500   
lr                    Logistic Regression    0.9891  0.9804  0.8880  0.1257   
ridge                    Ridge Classifier    0.9875  0.9731  0.8222  0.1042   
ada                  Ada Boost Classifier    0.9875  0.9723  0.8651  0.1094   
lda    

In [10]:
# =========================
# INSTALAR H2O
# =========================
%pip install h2o

# =========================
# IMPORTS
# =========================
import pandas as pd
import h2o
from h2o.automl import H2OAutoML

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

# =========================
# INICIAR H2O
# =========================
h2o.init()

# =========================
# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

# =========================
# PARA H2O
# =========================
train_h2o = h2o.H2OFrame(train_df)
test_h2o = h2o.H2OFrame(test_df)

train_h2o['Class'] = train_h2o['Class'].asfactor()
test_h2o['Class'] = test_h2o['Class'].asfactor()

x = train_h2o.columns[:-1]
y = 'Class'

# =========================
# AUTO ML
# =========================
aml = H2OAutoML(
    max_models=20,
    seed=42,
    balance_classes=True
)

aml.train(x=x, y=y, training_frame=train_h2o)

# =========================
# LEADERBOARD
# =========================
print("\n===== LEADERBOARD =====")
print(aml.leaderboard)

# =========================
# MÉTRICAS DE TODOS OS MODELOS
# =========================
leaderboard = aml.leaderboard.as_data_frame()

resultados = []

for model_id in leaderboard['model_id']:

    model = h2o.get_model(model_id)

    # previsão
    preds = model.predict(test_h2o).as_data_frame()

    y_true = test_df['Class'].values
    y_pred = preds['predict'].astype(int).values

    # métricas
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted')
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    average_precision = average_precision_score(y_true, y_pred, pos_label=1, average='weighted')

    resultados.append({
        "Modelo": model_id,
        "Tipo": model.algo,
        "Accuracy": round(acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-score": round(f1, 4),
        "F1-score Weighted": round(f1_weighted, 4),
        "average_precision": round(average_precision, 4)
    })

# =========================
# RESULTADO FINAL
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="average_precision", ascending=False))


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,24 days 20 hours 21 mins
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 19 days
H2O_cluster_name:,H2O_from_python_lannaces_gfr9u3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,458.9 Mb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
18:31:20.228: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%

===== LEADERBOARD =====
model